# EfficientNet with MixUp and CutMix

This notebook implements:
- **MixUp**: Blends two images and their labels
- **CutMix**: Cuts and pastes patches between images
- Both techniques improve generalization and reduce overfitting

In [1]:
from torch import optim
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import albumentations as Albu
import pandas as pd
from torch.utils.data.sampler import RandomSampler
from warmup_scheduler import GradualWarmupScheduler
from tqdm import tqdm
import os
import sys
sys.path.append('../../..')
from utils.dataset import PandasDataset
from utils.metrics import evaluation, format_metrics
from utils.models import EfficientNetApi

In [2]:
seed = 42
batch_size = 6
num_workers = 4
output_classes = 5
init_lr = 3e-4
warmup_factor = 2
warmup_epochs = 1
n_epochs = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

ROOT_DIR = '../../..'
data_dir = '../../../../dataset'
images_dir = os.path.join(data_dir, 'tiles')

Using device: cuda


## MixUp and CutMix Implementation

In [3]:
def mixup_data(x, y, alpha=1.0):
    '''Returns mixed inputs, pairs of targets, and lambda'''
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    # uniform
    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

def cutmix_data(x, y, alpha=1.0):
    '''Returns cutmix inputs, pairs of targets, and lambda'''
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    y_a, y_b = y, y[index]
    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
    
    # adjust lambda to exactly match pixel ratio
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size()[-1] * x.size()[-2]))
    return x, y_a, y_b, lam

print("MixUp and CutMix functions defined")

MixUp and CutMix functions defined


In [4]:
load_model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model = EfficientNetApi(model=load_model, output_dimensions=output_classes, dropout_rate=0.6)
model = model.to(device)

## Load Dataset

In [5]:
df_train_ = pd.read_csv(f"{ROOT_DIR}/data/train_5fold.csv")
df_train_.columns = df_train_.columns.str.strip()
train_indexes = np.where((df_train_['fold'] != 3))[0]
valid_indexes = np.where((df_train_['fold'] == 3))[0]

df_train = df_train_.loc[train_indexes]
df_val = df_train_.loc[valid_indexes]
df_test = pd.read_csv(f"{ROOT_DIR}/data/test.csv")

transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
])

train_dataset = PandasDataset(images_dir, df_train, transforms=transforms)
valid_dataset = PandasDataset(images_dir, df_val, transforms=None)
test_dataset = PandasDataset(images_dir, df_test, transforms=None)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(train_dataset)
)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(valid_dataset)
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(test_dataset)
)

## Custom Training Loop with MixUp/CutMix

In [6]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=init_lr/warmup_factor)
scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs - warmup_epochs)
scheduler = GradualWarmupScheduler(optimizer, multiplier=warmup_factor, total_epoch=warmup_epochs, after_scheduler=scheduler_cosine)

# Training parameters
mixup_alpha = 0.2
cutmix_alpha = 1.0
mixup_prob = 0.5  # probability to use mixup vs cutmix

best_kappa = 0.0
patience_counter = 0
patience = 5

log_file = open("logs/mixup-cutmix.txt", "w")

for epoch in range(1, n_epochs + 1):
    model.train()
    train_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{n_epochs}")
    for batch_idx, (inputs, targets, _) in enumerate(pbar):
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Randomly choose between MixUp and CutMix
        r = np.random.rand(1)
        if r < mixup_prob:
            inputs, targets_a, targets_b, lam = mixup_data(inputs, targets, mixup_alpha)
        else:
            inputs, targets_a, targets_b, lam = cutmix_data(inputs, targets, cutmix_alpha)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = mixup_criterion(loss_function, outputs, targets_a, targets_b, lam)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})
    
    train_loss /= len(train_loader)
    
    # Validation
    response = evaluation(model, valid_loader, device)
    val_kappa = response[0]['kappa']['mean']
    
    result = format_metrics(response[0])
    print(f"\nEpoch {epoch} | Train Loss: {train_loss:.4f}")
    print(result)
    
    log_file.write(f"epoch: {epoch} | lr: {optimizer.param_groups[0]['lr']:.7f} | "
                   f"Train loss: {train_loss} | Val Kappa: {val_kappa}\n")
    log_file.flush()
    
    # Save best model
    if val_kappa > best_kappa:
        best_kappa = val_kappa
        torch.save(model.state_dict(), "models/mixup-cutmix.pth")
        print(f"Saved best model with kappa: {best_kappa:.4f}")
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break
    
    scheduler.step()

log_file.close()
print(f"Training completed. Best kappa: {best_kappa:.4f}")

Epoch 1/50:   0%|          | 0/1204 [00:01<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 432.00 MiB. GPU 0 has a total capacity of 11.76 GiB of which 494.62 MiB is free. Process 2096041 has 9.00 GiB memory in use. Including non-PyTorch memory, this process has 1.15 GiB memory in use. Of the allocated memory 1.02 GiB is allocated by PyTorch, and 18.45 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Test

In [ ]:
model.load_state_dict(torch.load("models/mixup-cutmix.pth"))
response = evaluation(model, test_loader, device)
result = format_metrics(response[0])
print("\n=== TEST RESULTS ===")
print(result)